# Revision R2C4.a | R2C2.c: Dataset audit analysis

Response to:

> R2C4.a: The large improvement from one-hot source pre-training in Figure 4A-D could reflect mutation spectra, database ascertainment, `gene composition`, or class-specific sequence context rather than reusable pathogenic biology. 

> R2C2.c: The source-domain overlap audit should also cover `genes`, `exons`, `normalized alleles`, `overlapping 129-bp windows`, and `near-identical sequence clusters`; genome-wide `rare-variant analyses` illustrate how strongly gene and annotation composition can shape an apparent signal.

---

For ID mapping:

nohup wget -c https://ftp.ncbi.nlm.nih.gov/gene/DATA/gene2ensembl.gz &

# 1 Gene composition analysis

In [ ]:
from pathlib import Path
import pandas as pd

TRANSCRIPTS = Path("../dataset/x-source-data/transcript_id_info_summary.txt")
MISSENSE =  Path("../dataset/b-output/missense_dataset/")
SYNONYMOUS =  Path("../dataset/b-output/synonymous_dataset/")

INPUTS = {
	"missense": {
		"benign": MISSENSE / "missense.bn.with_transcript.sanity.check.3.noDMq.txt",
		"pathogenic": MISSENSE / "missense.pt.with_transcript.sanity.check.3.noDMq.txt"},
	"synonymous": {
		"benign": SYNONYMOUS / "train_neg.txt",
		"pathogenic": SYNONYMOUS / "train_pos.txt"}}

# read in gene table
gene_df = pd.read_csv(TRANSCRIPTS, sep='\t')
gene_df.info()

## 1.1 Synonymous variants

In [ ]:
syn_tid = set()
for k, v in INPUTS['synonymous'].items():
    tid = set(pd.read_csv(v, sep='\t', usecols=['transcript_id'])['transcript_id'])
    syn_tid.update(tid)
print(f"[INFO] syn transcript amount: {len(syn_tid)}")

In [ ]:
# Count genes containing synonymous variants
syn_tid_df = gene_df[gene_df['transcript_id'].isin(syn_tid)]
syn_gid = set(syn_tid_df['gene_id'])
print(f"[INFO] Number of genes containing synonymous variants: {len(syn_gid)}")

## 1.2 Missense variants

In [ ]:
# Count transcripts containing missense variants
mis_tid = set()
for k, v in INPUTS['missense'].items():
    tid = set(pd.read_csv(v, sep='\t', usecols=['refseq_id'])['refseq_id'])
    mis_tid.update(tid)
print(f"[INFO] Number of transcripts containing missense variants: {len(mis_tid)}")

In [ ]:
# Count genes containing missense variants
mis_tid_df = gene_df[gene_df['refseq_id'].isin(mis_tid)]
mis_gid = set(mis_tid_df['gene_id'])
print(f"[INFO] Number of genes containing missense variants: {len(mis_gid)}")

## 1.3 Non-coding variants

In [ ]:
# Load noncoding variants
import pandas as pd

file_path = "../dataset/b-output/noncoding_dataset"
pt_snv = pd.read_csv(f"{file_path}/noncoding.pt.rmSyn.txt", sep='\t', usecols=['variant38'])
bn_snv = pd.read_csv(f"{file_path}/noncoding.bn.rmSyn.txt", sep='\t', usecols=['variant38'])
noncoding_snv = pd.concat([pt_snv, bn_snv], ignore_index=True)
noncoding_snv.drop_duplicates(subset='variant38', inplace=True)
print(f"[INFO] Number of noncoding variants: {len(noncoding_snv)}")


In [ ]:
# Count transcripts containing noncoding variants
noncoding_tid = set(noncoding_snv['refseq_id'])
print(f"[INFO] Number of transcripts containing non-coding variants: {len(noncoding_tid)}")

In [ ]:
# Count genes containing non-coding variants
nc_tid_df = gene_df[gene_df['refseq_id'].isin(noncoding_tid)]
nc_gid = set(nc_tid_df['gene_id'])
print(f"[INFO] Number of genes containing noncoding variants: {len(nc_gid)}")

## 1.4 Gene-overlap analysis


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn3

# Ensure that inputs are sets
syn = set(syn_gid)
mis = set(mis_gid)
nc = set(nc_gid)

# Seven mutually exclusive regions
regions = {
    "Synonymous only": syn - mis - nc,
    "Missense only": mis - syn - nc,
    "Noncoding only": nc - syn - mis,
    "Synonymous & Missense only": (syn & mis) - nc,
    "Synonymous & Noncoding only": (syn & nc) - mis,
    "Missense & Noncoding only": (mis & nc) - syn,
    "Synonymous & Missense & Noncoding": syn & mis & nc,
}

# Report statistics for each region
stats = pd.DataFrame(
    [(name, len(genes)) for name, genes in regions.items()],
    columns=["Region", "Gene_count"],
)
print(stats.to_string(index=False))

print(f"\nTotal synonymous genes: {len(syn):,}")
print(f"Total missense genes: {len(mis):,}")
print(f"Total non-coding genes: {len(nc):,}")
print(f"Total genes in the union: {len(syn | mis | nc):,}")

# Draw the Venn diagram
fig, ax = plt.subplots(figsize=(7, 6))

v = venn3(
    subsets=(syn, mis, nc),
    set_labels=(
        f"Synonymous\nvariant\n(n={len(syn):,})",
        f"Missense\nvariant\n(n={len(mis):,})",
        f"Noncoding\nvariant\n(n={len(nc):,})",
    ),
    # set_colors=("#66C2A5", "#FC8D62", "#8DA0CB"),
    # set_colors=("#b21e34", "#dba521", "#23a4a7"),
    # set_colors=("#faa44d", "#71cdda", "#1271c1"),
    set_colors=( "#ea3a47", "#6396f2", "#010101",),
    alpha=0.9,
    subset_label_formatter=lambda x: f"{x:,}",
    ax=ax,
)

import numpy as np

# Set-label font size
for label in v.set_labels:
    if label is not None:
        label.set_fontsize(14)

# Region-label font sizes and colors
for region_id in ("100", "010", "110", "001", "101", "011", "111"):
    label = v.get_label_by_id(region_id)
    patch = v.get_patch_by_id(region_id)

    if label is None or patch is None:
        continue

    label.set_fontsize(13)

    # Blend the region color with the axes background according to alpha
    rgba = np.asarray(patch.get_facecolor())
    background = np.asarray(ax.get_facecolor()[:3])
    rgb = rgba[:3] * rgba[3] + background * (1 - rgba[3])

    # Convert sRGB to linear RGB and calculate relative luminance
    linear_rgb = np.where(
        rgb <= 0.04045,
        rgb / 12.92,
        ((rgb + 0.055) / 1.055) ** 2.4,
    )
    luminance = np.dot(linear_rgb, [0.2126, 0.7152, 0.0722])

    # Select black or white for higher contrast with the background
    black_contrast = (luminance + 0.05) / 0.05
    white_contrast = 1.05 / (luminance + 0.05)

    label.set_color(
        "white" if white_contrast > black_contrast else "black"
    )


# ax.set_title("Overlap of genes from source and target domains")
plt.tight_layout()
plt.savefig(
	'./plot_result/Gene_venn_domains.jpg',  # File name
	dpi=300,           # Resolution in dots per inch
	bbox_inches='tight', # Tight layout without excess whitespace
)
plt.show()